# Mouse 262.478 and the institution name

Every raw file and `Subject:` field for this DMS Inhibitory Group 2 mouse reads `262` or `262.478`. Only the file name `2021-10-29_262_259.478`, which also has a copy in `Scrambled`, reads 262.259. The demographics sheet originally listed 262.478; in May 2024 the conversion's copy of the sheet was changed to 262.259 to match that one file name, which made 262.478 "missing" and dropped the mouse's other sessions. DANDI:000971 holds only the Scrambled copy of the 10/29 run, labeled NpHRScrambled.

With the sheet row set back to 262.478, this notebook converts the mouse's sessions and compares the 10/29 run with the published file. It also shows the institution name, which was misspelled in every published file.

In [1]:
from pathlib import Path
import shutil

import h5py
import numpy as np
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import (
    SUBJECTS_TO_SKIP,
    opto_to_nwb,
    get_no_port_entry_duration_sessions,
    get_session_key_from_kwargs,
)
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/subject_labels")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)

In [2]:
demographics = pd.read_excel(data_dir_path / "MouseDemographicsCorrected.xlsx", sheet_name="Mouse Demographics", dtype={"Mouse ID": str})
demographics[demographics["Mouse ID"].str.startswith("262")][["Mouse ID", "Experiment", "Sex", "Treatment"]]

,Mouse ID,Experiment,Sex,Treatment
149,262.478,DMS Inhibitory Group 2,Male,Inhibitory stimulation on rewarded nosepokes (...


In [3]:
session_to_nwb_kwargs_per_session = [
    session_to_nwb_kwargs
    for session_to_nwb_kwargs in opto_to_nwb(
        data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False
    )
    if session_to_nwb_kwargs["subject_id"].startswith("262")
]
pd.DataFrame(
    [
        dict(
            subject_id=session_to_nwb_kwargs["subject_id"],
            treatment=session_to_nwb_kwargs["optogenetic_treatment"],
            file=session_to_nwb_kwargs["behavior_file_path"].relative_to(data_dir_path / "Opto Experiments").as_posix(),
            skipped=session_to_nwb_kwargs["subject_id"] in SUBJECTS_TO_SKIP,
        )
        for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session
    ]
).value_counts().to_frame("sessions")

sessions
subject_id treatment     file                                               skipped          
262.478    NpHR          DMS Inhibitory/Group 2/NpHr/262.478                False          18
262.259    NpHRScrambled DMS Inhibitory/Group 2/Scrambled/2021-10-29_262... True            1
262.478    NpHR          DMS Inhibitory/Group 2/NpHr/2021-10-25_262.478     False           1
                         DMS Inhibitory/Group 2/NpHr/2021-10-26_262.478     False           1
                         DMS Inhibitory/Group 2/NpHr/2021-10-27_262.478     False           1
                         DMS Inhibitory/Group 2/NpHr/2021-10-29_262_259.478 False           1

In [4]:
selected_kwargs = [
    session_to_nwb_kwargs
    for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session
    if session_to_nwb_kwargs["subject_id"] not in SUBJECTS_TO_SKIP
]
no_port_entry_duration_sessions = get_no_port_entry_duration_sessions(
    port_entry_file_path=output_dir_path / "sessions_without_port_entry_durations.yaml",
    session_to_nwb_args_per_session=selected_kwargs,
)
for session_to_nwb_kwargs in selected_kwargs:
    if get_session_key_from_kwargs(session_to_nwb_kwargs) in no_port_entry_duration_sessions:
        session_to_nwb_kwargs["has_port_entry_durations"] = False
    session_to_nwb(**session_to_nwb_kwargs)
print(f"{len(list(output_dir_path.glob('*.nwb')))} files converted")

Reading port entry durations:   0%|          | 0/22 [00:00<?, ?it/s]

Reading port entry durations:  55%|█████▍    | 12/22 [00:00<00:00, 106.21it/s]

Reading port entry durations: 100%|██████████| 22/22 [00:00<00:00, 88.63it/s] 

22 files converted


In [5]:
def summarize(file_path: Path) -> dict:
    with h5py.File(file_path, "r") as file:
        return dict(
            file=file_path.name,
            subject_id=file["general/subject/subject_id"][()].decode(),
            sex=file["general/subject/sex"][()].decode(),
            stimulus_notes=file["general/stimulus"][()].decode(),
            institution=file["general/institution"][()].decode(),
        )


def read_behavior_datasets(file_path: Path) -> dict[str, np.ndarray]:
    datasets = {}
    with h5py.File(file_path, "r") as file:
        for group in ("processing/behavior", "stimulus"):
            if group in file:
                file[group].visititems(
                    lambda name, item: datasets.__setitem__(f"{group}/{name}", item[()]) if isinstance(item, h5py.Dataset) else None
                )
    return datasets


(published_file_path,) = (published_dir_path / "sub-262-259").glob("*.nwb")
(new_file_path,) = output_dir_path.glob("*2021-10-29T15-11-08*.nwb")
pd.DataFrame([dict(version="published", **summarize(published_file_path)), dict(version="fixed", **summarize(new_file_path))])

,version,file,subject_id,sex,stimulus_notes,institution
0,published,sub-262-259_ses-Opto-DMS-Inhibitory-NpHRScramb...,262.259,M,Inhibitory stimulation on rewarded nosepokes (...,Northwestern Unitersity
1,fixed,sub-262.478_ses-Opto-DMS-Inhibitory-NpHR-2021-...,262.478,M,Inhibitory stimulation on rewarded nosepokes (...,Northwestern University


In [6]:
published_datasets = read_behavior_datasets(published_file_path)
new_datasets = read_behavior_datasets(new_file_path)
print("same datasets:", set(published_datasets) == set(new_datasets))
print("same values:", all(np.array_equal(published_datasets[name], new_datasets[name]) for name in new_datasets))

same datasets: True
same values: True
